# 05 · Hyperventilation — breathing (depth & rate)

Supports the report's "hyperventilation: half-confirmed" finding: the belt
**amplitude** rises during the challenge, but the **rate** does not increase.

In [1]:
import warnings, pathlib, sys
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
_c = pathlib.Path.cwd()
REPO = next(p for p in [_c] + list(_c.parents) if (p / "analysis").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from analysis import dataset, eegkit as K
DATA_DIR = None
recs = dataset.discover(DATA_DIR)
_gi, LABEL = {}, {}
for _r in recs:
    _g = _r["group"]; _gi[_g] = _gi.get(_g, 0) + 1
    LABEL[_r["key"]] = ("exp" if _g == "E" else "ctrl") + str(_gi[_g])
def lbl(r): return LABEL[r["key"]]
FIG = REPO / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def ec_window(r):
    st = K.get_stages(K.load(r["path"]), r["key"])
    return [s for s in st if s[0].startswith("rest_close")][0]
def get_iaf(r):
    raw = K.load(r["path"]); ec = ec_window(r)
    _, live = K.get(raw, ec[1], ec[2]); return K.iaf(raw, live, ec[1], ec[2])
def stage_of(r, nm):
    st = K.get_stages(K.load(r["path"]), r["key"]); return [s for s in st if s[0] == nm]
print(f"discovered {len(recs)} recordings ->", {lbl(r): r["group"] for r in recs})


discovered 4 recordings -> {'exp1': 'E', 'exp2': 'E', 'ctrl1': 'C', 'ctrl2': 'C'}


In [2]:
from scipy.signal import find_peaks
def belt_rate_amp(raw, ch, t0, t1):
    fs = raw.info["sfreq"]
    d = K.bp(raw.get_data()[raw.ch_names.index(ch)] * 1e3, fs, 0.05, 0.5)   # mV
    seg = d[int(t0*fs):int(t1*fs)]
    if seg.std() < 0.05:
        return np.nan, np.nan
    pk,_ = find_peaks(seg, distance=int(0.7*fs), prominence=0.3*np.std(seg))
    return len(pk)/((t1-t0)/60.0), seg.std()
rows=[]
for r in recs:
    raw = K.load(r["path"])
    rc,_ = K.resp_channel(raw, 60, min(240, raw.n_times/raw.info["sfreq"]))
    st = {n:(a,b) for n,a,b in K.get_stages(raw, r["key"])}
    row = dict(rec=lbl(r), group=r["group"], belt=rc or "none")
    for nm in ("rest_open1","hypervent","speech"):
        if rc and nm in st:
            rt,amp = belt_rate_amp(raw, rc, *st[nm]); row[f"rate_{nm}"]=round(rt,1); row[f"amp_{nm}"]=round(amp,2)
    if rc and "hypervent" in st and "rest_open1" in st and row.get("amp_rest_open1",0):
        row["amp_x_vs_rest"] = round(row["amp_hypervent"]/row["amp_rest_open1"],2)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nBelt amplitude x2-3 during HV (deeper breaths) but rate does NOT rise -> task-compliance caveat.")


  rec group belt  rate_rest_open1  amp_rest_open1  rate_hypervent  amp_hypervent  rate_speech  amp_speech  amp_x_vs_rest
 exp1     E Pesp              NaN             NaN             NaN            NaN          NaN         NaN            NaN
 exp2     E Pesp              0.3            2.31             NaN            NaN          NaN         NaN            NaN
ctrl1     C none              NaN             NaN             NaN            NaN          NaN         NaN            NaN
ctrl2     C none              NaN             NaN             NaN            NaN          NaN         NaN            NaN

Belt amplitude x2-3 during HV (deeper breaths) but rate does NOT rise -> task-compliance caveat.
